# SAE Steering v2: All 3 Methods (Colab GPU)

Full generation pipeline with **SAE steering** using `intervention_v2.pt`.
Only **most likely answer** is generated (greedy decoding, no sampling).

**Three steering methods:**

| Method | Flag | How it works |
|--------|------|--------------|
| **EMD** | `sae_emd` | `f' = f + α·δ` (Cohen's d weighted), `h' = decode(f') + err` |
| **Projected VUF** | `sae_projected_vuf` | Same, but δ = SAE-projected VUF masked to consensus features |
| **Clamp** | `sae_clamp` | Uncertainty features → push up, certainty features → suppress |

**Files to upload:**
1. **test.csv** — CSV with `question`, `verbal_uncertainty`, `sentence_semantic_entropy`
2. **intervention_v2.pt** — config from `build_intervention_config_v2.py`

**Workflow:**
1. Debug: first N questions with **all 3 methods**, answers printed
2. Full generation with **all 3 methods** sequentially, α quantized
3. Save 3 separate JSONL files (name includes method and alpha_max) + download

## 0. Check GPU

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print(f"Python: {sys.version}")

## 1. Clone repository and install dependencies

Clone takes seconds; pip takes 2–5 minutes (transformers + sae-lens + accelerate).

In [ ]:
import os, sys, subprocess

GIT_URL = "https://github.com/SadreevAmir/sae-muc.git"
GIT_BRANCH = "main"
REPO_DIR = "/content/sae-muc"

sae_pkg = os.path.join(REPO_DIR, "sae_muc")
if not os.path.isdir(sae_pkg):
    if os.path.isdir(REPO_DIR):
        subprocess.run(["rm", "-rf", REPO_DIR], check=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("-> Installing dependencies ...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "numpy>=2.0.0,<2.1",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U", "--no-cache-dir",
    "transformers>=4.40", "accelerate",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "sae-lens>=6.0", "pandas", "tqdm", "jsonlines", "huggingface_hub",
])

import torch
assert torch.cuda.is_available(), "GPU not available — change Runtime type to GPU!"
print(f"\nREPO: {REPO_DIR}")
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. (Optional) Google Drive

If files are on Drive — mount it and set paths in section 4.

In [ ]:
MOUNT_DRIVE = False  # @param {type:"boolean"}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted at /content/drive")
else:
    print("Drive not mounted — will use file upload in section 3.")

## 3. Upload files

Upload **two files**:
1. **test.csv** — CSV with columns `question`, `verbal_uncertainty`, `sentence_semantic_entropy`
2. **intervention_v2.pt** — intervention config (from `build_intervention_config_v2.py`)

If files are already available — set `UPLOAD_FILES = False` and specify paths in section 4.

In [ ]:
import os

UPLOAD_FILES = True  # @param {type:"boolean"}

UPLOAD_DIR = "/content/uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

_uploaded_csv = None
_uploaded_pt = None

if UPLOAD_FILES:
    from google.colab import files

    print("=" * 60)
    print("Step 1/2: Upload test.csv")
    print("  Required columns: question, verbal_uncertainty, sentence_semantic_entropy")
    print("=" * 60)
    up1 = files.upload()
    for fn, data in up1.items():
        dst = os.path.join(UPLOAD_DIR, fn)
        with open(dst, "wb") as f:
            f.write(data)
        if fn.endswith(".csv"):
            _uploaded_csv = dst
            print(f"  -> Saved CSV: {dst}")

    print()
    print("=" * 60)
    print("Step 2/2: Upload intervention_v2.pt")
    print("=" * 60)
    up2 = files.upload()
    for fn, data in up2.items():
        dst = os.path.join(UPLOAD_DIR, fn)
        with open(dst, "wb") as f:
            f.write(data)
        if fn.endswith(".pt"):
            _uploaded_pt = dst
            print(f"  -> Saved PT: {dst}")

    if _uploaded_csv:
        print(f"\nCSV: {_uploaded_csv}")
    if _uploaded_pt:
        print(f"PT:  {_uploaded_pt}")
else:
    print("Upload skipped — set paths manually in section 4.")

## 4. Configuration

| Parameter | Description |
|---|---|
| `TEST_CSV` | CSV with questions |
| `INTERVENTION_PT` | `.pt` file from `build_intervention_config_v2.py` |
| `ALPHA_MAX` | Maximum intervention strength |
| `ALPHA_BINS` | Number of α quantization bins (step = max/bins) |
| `N_DEBUG` | How many first questions to run for debugging (all methods) |
| `DEBUG_ALPHA` | Fixed α for the debug run |
| `OUTPUT_DIR` | Output folder for JSONL (filename = `{method}_alpha{ALPHA_MAX}.jsonl`) |

In [ ]:
# ── File paths (auto-filled from upload, or set manually) ──
TEST_CSV         = _uploaded_csv or "/content/uploads/test.csv"            # @param {type:"string"}
INTERVENTION_PT  = _uploaded_pt  or "/content/uploads/intervention_v2.pt"  # @param {type:"string"}

# ── Model ──
MODEL_NAME       = "Mistral-7B-Instruct-v0.3"  # @param {type:"string"}
SAE_DTYPE        = "float32"                    # @param ["float32", "float16", "bfloat16"]

# ── Alpha quantization ──
ALPHA_MAX        = 50.0   # @param {type:"number"}
ALPHA_BINS       = 4      # @param {type:"integer"}

# ── Generation ──
GEN_BATCH_SIZE   = 4      # @param {type:"integer"}
APPLY_DURING_GEN = True   # @param {type:"boolean"}

# ── Debug settings ──
N_DEBUG          = 5      # @param {type:"integer"}
DEBUG_ALPHA      = 2.0    # @param {type:"number"}

# ── Output ──
OUTPUT_DIR       = "/content/sae_steering_results"  # @param {type:"string"}

# ── Validate ──
import os, pandas as pd
assert os.path.isfile(TEST_CSV),        f"CSV not found: {TEST_CSV}"
assert os.path.isfile(INTERVENTION_PT), f"PT not found: {INTERVENTION_PT}"

df_preview = pd.read_csv(TEST_CSV)
required_cols = {"question", "verbal_uncertainty", "sentence_semantic_entropy"}
missing = required_cols - set(df_preview.columns)
assert not missing, f"CSV is missing columns: {missing}"

print(f"CSV:             {TEST_CSV}  ({len(df_preview)} rows)")
print(f"Intervention:    {INTERVENTION_PT}")
print(f"Model:           {MODEL_NAME}")
print(f"Alpha:           max={ALPHA_MAX}, bins={ALPHA_BINS}, step={ALPHA_MAX/ALPHA_BINS:.2f}")
print(f"Debug:           {N_DEBUG} questions, alpha={DEBUG_ALPHA}")
print(f"Output dir:      {OUTPUT_DIR}")
print(f"  File pattern:  {{method}}_alpha{ALPHA_MAX}.jsonl")
print()
df_preview.head(5)

## 5. Hugging Face Login

A token with model access is required (for Mistral — accept the license on HF first).

In [ ]:
from huggingface_hub import login
login()

## 6. Load model, SAE and intervention config

Loads:
- `intervention_v2.pt` — defines SAE release, layers, data for all 3 methods
- SAE for each layer (shared across methods)
- LLM (~14 GB fp16)
- Data from CSV, computes quantized α values

In [ ]:
import json, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sae_lens import SAE

import sys, os
REPO_DIR = "/content/sae-muc"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from sae_muc.hooks import (
    register_sae_latent_hooks,
    register_sae_clamp_hooks,
    clear_sae_latent_hooks,
)
from sae_muc.generation import generate_lines_for_batch
from sae_muc.prompts_mini import make_sentence_user_content

torch.manual_seed(42)
np.random.seed(42)

MAX_SE = 2.302585092994045

# ── Resolve model name ──
if "Mistral" in MODEL_NAME:
    full_model_name = f"mistralai/{MODEL_NAME}"
elif "Llama" in MODEL_NAME:
    full_model_name = f"meta-llama/{MODEL_NAME}"
elif "Qwen" in MODEL_NAME:
    full_model_name = f"Qwen/{MODEL_NAME}"
else:
    full_model_name = MODEL_NAME

# ── Load CSV ──
print("Loading CSV ...")
df = pd.read_csv(TEST_CSV)
questions = df["question"].astype(str).tolist()
vu = df["verbal_uncertainty"].to_numpy(dtype=np.float64)
se = df["sentence_semantic_entropy"].to_numpy(dtype=np.float64)

print(f"Questions: {len(questions)}")

messages = [[{"role": "user", "content": make_sentence_user_content(q)}] for q in questions]

# ── Load intervention_v2.pt ──
print("\nLoading intervention config ...")
intervention_data = torch.load(INTERVENTION_PT, map_location="cpu", weights_only=False)
release = intervention_data["release"]
raw_layers = intervention_data["layers"]
meta = intervention_data.get("meta", {})

print(f"  Release: {release}")
print(f"  Meta: {meta}")
print(f"  Layers: {sorted(int(k) for k in raw_layers.keys())}")

# ── Detect available methods ──
available_methods = []
sample_layer = next(iter(raw_layers.values()))
if "method_emd" in sample_layer:
    available_methods.append("sae_emd")
if sample_layer.get("method_projected_vuf") is not None:
    available_methods.append("sae_projected_vuf")
if "method_clamp" in sample_layer:
    available_methods.append("sae_clamp")

print(f"  Available methods: {available_methods}")
assert len(available_methods) > 0, "No steering methods found in the intervention config!"

# ── Helper: extract method-specific data from loaded intervention ──
def extract_method_data(data: dict, method: str) -> dict:
    """Extract per-layer data for a given method from raw intervention dict."""
    layers = {}
    for k, v in data["layers"].items():
        hf_layer = int(k)
        entry = {"sae_id": v["sae_id"]}
        if method == "sae_emd":
            entry["delta"] = v["method_emd"]["delta"]
        elif method == "sae_projected_vuf":
            pvuf = v.get("method_projected_vuf")
            if pvuf is None:
                continue
            entry["delta"] = pvuf["delta"]
        elif method == "sae_clamp":
            clamp = v["method_clamp"]
            unc_idx = clamp["uncertainty_features"]
            cert_idx = clamp["certainty_features"]
            target_vals = clamp["target_uncertain_values"]
            entry["clamp_config"] = {
                "unc_indices": torch.tensor(unc_idx, dtype=torch.long),
                "unc_targets": torch.tensor(
                    [target_vals[i] for i in unc_idx], dtype=torch.float32
                ),
                "cert_indices": torch.tensor(cert_idx, dtype=torch.long),
            }
        layers[hf_layer] = entry
    return layers

# ── Load SAEs (shared across all methods) ──
print("\nLoading SAEs ...")
layer_to_sae = {}
for k, v in raw_layers.items():
    hf_layer = int(k)
    sae_id = v["sae_id"]
    t0 = time.time()
    sae = SAE.from_pretrained(release=release, sae_id=sae_id, device="cpu", dtype=SAE_DTYPE)
    layer_to_sae[hf_layer] = sae
    print(f"  Layer {hf_layer} ({sae_id}): d_sae={sae.cfg.d_sae}, loaded in {time.time()-t0:.1f}s")

hook_layers = sorted(layer_to_sae.keys())

# ── Print method summaries ──
for method in available_methods:
    method_layers = extract_method_data(intervention_data, method)
    if method == "sae_clamp":
        for l, info in method_layers.items():
            cc = info["clamp_config"]
            print(f"  {method} L{l}: {cc['unc_indices'].numel()} unc features, {cc['cert_indices'].numel()} cert features")
    else:
        for l, info in method_layers.items():
            nz = int((info["delta"] != 0).sum().item())
            print(f"  {method} L{l}: {nz} nonzero features in delta")

# ── Load LLM ──
print(f"\nLoading {full_model_name} (fp16, device_map=auto) ...")
from transformers import AutoModelForCausalLM, AutoTokenizer

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    full_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(full_model_name, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Model loaded in {time.time()-t0:.1f}s")
print(f"\nReady: {len(questions)} questions, {len(hook_layers)} SAE layers, {len(available_methods)} methods")

## 7. Debug: first N questions, all methods

Runs the first `N_DEBUG` questions with **baseline + all available methods** at fixed `DEBUG_ALPHA`.
Results are printed for visual inspection.

This is a quick sanity check — make sure the model responds and steering works.

In [ ]:
from tqdm.auto import tqdm

# ── Recompute alphas from current config ──
alpha_step = ALPHA_MAX / float(ALPHA_BINS)
raw_alpha = (se / MAX_SE - vu) * ALPHA_MAX
raw_alpha = np.clip(raw_alpha, 0.0, ALPHA_MAX)
alphas = np.clip(np.round(raw_alpha / alpha_step) * alpha_step, 0.0, ALPHA_MAX)
alphas = np.round(alphas, 6)

n_dbg = min(N_DEBUG, len(questions))
dbg_questions = questions[:n_dbg]
dbg_messages = messages[:n_dbg]

print(f"Debug: {n_dbg} questions, alpha={DEBUG_ALPHA}")
print(f"Available methods: {available_methods}")
print("=" * 80)

# ── Helper to register hooks for any method ──
def register_hooks_for_method(method, alpha):
    clear_sae_latent_hooks(model)
    method_layers = extract_method_data(intervention_data, method)
    if method == "sae_clamp":
        layer_to_clamp = {l: info["clamp_config"] for l, info in method_layers.items()}
        register_sae_clamp_hooks(
            model, layer_to_sae, layer_to_clamp, hook_layers, alpha,
            apply_during_generation=APPLY_DURING_GEN,
        )
    else:
        layer_to_delta = {l: info["delta"] for l, info in method_layers.items()}
        register_sae_latent_hooks(
            model, layer_to_sae, layer_to_delta, hook_layers, alpha,
            apply_during_generation=APPLY_DURING_GEN,
        )

# ── Baseline ──
print("\nGenerating BASELINE ...")
clear_sae_latent_hooks(model)
baseline_lines = generate_lines_for_batch(
    model, tokenizer, dbg_questions, dbg_messages, 0.0,
    greedy_only=True,
)

# ── Each method ──
debug_results = {}
for method in available_methods:
    print(f"Generating {method} (alpha={DEBUG_ALPHA}) ...")
    register_hooks_for_method(method, DEBUG_ALPHA)
    lines = generate_lines_for_batch(
        model, tokenizer, dbg_questions, dbg_messages, DEBUG_ALPHA,
        greedy_only=True,
    )
    debug_results[method] = lines
    clear_sae_latent_hooks(model)

# ── Display results ──
print("\n" + "=" * 80)
print("DEBUG RESULTS")
print("=" * 80)

for i in range(n_dbg):
    q = dbg_questions[i]
    print(f"\n{'─' * 80}")
    print(f"Q{i+1}: {q}")
    print(f"     VU={vu[i]:.3f}  SE={se[i]:.3f}  alpha_quantized={alphas[i]:.2f}")
    print(f"{'─' * 80}")
    base_ans = baseline_lines[i]["most_likely_answer"][:200]
    print(f"  {'BASELINE':<22s} {base_ans}")
    for method in available_methods:
        ans = debug_results[method][i]["most_likely_answer"][:200]
        print(f"  {method.upper():<22s} {ans}")

print(f"\n{'=' * 80}")
print("Debug done. Review answers above before running full generation.")

## 8. Full generation (all 3 methods)

Generation for **all** questions with **all available methods** sequentially (most likely answer only).

Questions are grouped by quantized α — hooks are registered once per group.
Results: `all_results[method] = [...]`, saved as `{method}_alpha{ALPHA_MAX}.jsonl`.

In [ ]:
from tqdm.auto import tqdm
import json, time

# ── Recompute alphas from current config (ALPHA_MAX, ALPHA_BINS) ──
alpha_step = ALPHA_MAX / float(ALPHA_BINS)
raw_alpha = (se / MAX_SE - vu) * ALPHA_MAX
raw_alpha = np.clip(raw_alpha, 0.0, ALPHA_MAX)
alphas = np.clip(np.round(raw_alpha / alpha_step) * alpha_step, 0.0, ALPHA_MAX)
alphas = np.round(alphas, 6)

print(f"Full generation: {len(questions)} questions x {len(available_methods)} methods")
print(f"Alpha: max={ALPHA_MAX}, bins={ALPHA_BINS}, step={alpha_step:.4f}")
print(f"Apply during generation: {APPLY_DURING_GEN}")
print(f"Mode: greedy only (most_likely_answer)")
print()

by_alpha = {}
for i, a in enumerate(alphas.tolist()):
    by_alpha.setdefault(float(a), []).append(i)

print(f"Alpha groups: {len(by_alpha)}")
for a in sorted(by_alpha.keys()):
    print(f"  alpha={a:8.4f}: {len(by_alpha[a]):4d} questions")
print()

batch_size = max(1, GEN_BATCH_SIZE)
all_results = {}

for method in available_methods:
    print(f"\n{'=' * 70}")
    print(f"Method: {method}")
    print(f"{'=' * 70}")
    t0 = time.time()

    results = [None] * len(questions)

    for alpha in sorted(by_alpha.keys()):
        clear_sae_latent_hooks(model)
        if alpha > 0:
            register_hooks_for_method(method, alpha)

        idxs = by_alpha[alpha]
        desc = f"{method} a={alpha:.1f} ({len(idxs)} qs)"

        for j in tqdm(range(0, len(idxs), batch_size), desc=desc):
            chunk_idx = idxs[j : j + batch_size]
            batch_q = [questions[k] for k in chunk_idx]
            batch_m = [messages[k] for k in chunk_idx]
            lines = generate_lines_for_batch(
                model, tokenizer, batch_q, batch_m, alpha, greedy_only=True,
            )
            for local_i, global_i in enumerate(chunk_idx):
                results[global_i] = lines[local_i]

        clear_sae_latent_hooks(model)
        torch.cuda.empty_cache()

    assert all(r is not None for r in results), f"Some rows missing for {method}!"
    all_results[method] = results
    print(f"  Done {method}: {len(results)} rows in {time.time()-t0:.0f}s")

print(f"\nAll methods done: {list(all_results.keys())}")

## 9. Save and download

In [ ]:
import json
from pathlib import Path

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

saved_paths = {}
for method, results in all_results.items():
    fname = f"{method}_alpha{ALPHA_MAX}.jsonl"
    out_path = out_dir / fname
    with open(out_path, "w", encoding="utf-8") as f:
        for item in results:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    saved_paths[method] = out_path
    print(f"Saved: {out_path}  ({len(results)} lines)")

# Preview
for method, results in all_results.items():
    print(f"\n{'=' * 60}")
    print(f"Preview: {method} (first 3 rows)")
    print(f"{'=' * 60}")
    for i, item in enumerate(results[:3]):
        print(f"  [{i}] alpha={item['alpha']:.4f}")
        print(f"      Q: {item['question'][:80]}")
        ans = item['most_likely_answer'][:120] if item['most_likely_answer'] else '(empty)'
        print(f"      A: {ans}")

# Stats
print(f"\n{'=' * 60}")
for method, results in all_results.items():
    alpha_vals = [r["alpha"] for r in results]
    n_base = sum(1 for a in alpha_vals if a == 0)
    n_steer = sum(1 for a in alpha_vals if a > 0)
    print(f"{method}: {len(results)} rows  (baseline={n_base}, steered={n_steer})")

# Download all
try:
    from google.colab import files
    for method, p in saved_paths.items():
        files.download(str(p))
        print(f"Download started: {p.name}")
except ImportError:
    print(f"\nNot in Colab — files saved in: {out_dir}")

## 10. Quick results analysis

In [ ]:
import pandas as pd
import numpy as np

print(f"Alpha config: max={ALPHA_MAX}, bins={ALPHA_BINS}, step={ALPHA_MAX/ALPHA_BINS:.4f}\n")

for method, results in all_results.items():
    df_out = pd.DataFrame(results)
    df_out["answer_len"] = df_out["most_likely_answer"].str.len()
    df_out["has_answer"] = df_out["most_likely_answer"].str.len() > 0

    print(f"{'=' * 60}")
    print(f"Method: {method}")
    print(f"{'=' * 60}")
    print("Per-alpha group stats:")
    print(df_out.groupby("alpha").agg(
        count=("question", "count"),
        mean_answer_len=("answer_len", "mean"),
        pct_has_answer=("has_answer", "mean"),
    ).to_string())
    print(f"\nOverall: {len(df_out)} rows, "
          f"answer_len: mean={df_out['answer_len'].mean():.0f}, "
          f"min={df_out['answer_len'].min()}, max={df_out['answer_len'].max()}")
    print()